In [20]:
import xarray as xr
import glob
from dask_jobqueue import PBSCluster
from dask.distributed import Client
import numpy as np

In [21]:
def crop_ds(ds, border=[-90, 90, -180, 180]):
    ds.coords['longitude'] = (ds.coords['longitude'] + 180) % 360 - 180
    ds = ds.sortby(ds.longitude)
    ds = ds.sortby(ds.latitude)
    ds_crop = ds.sel(latitude=slice(border[0], border[1]), longitude=slice(border[2], border[3]))
    return ds_crop


def ONI(sstanom):
    sstanom = crop_ds(sstanom, [-5, 5, -170, -120])
    index = sstanom.mean(('longitude', 'latitude')).rolling(valid_time=3).mean('valid_time')
    return index


def ATL3(sstanom):
    sstanom = crop_ds(sstanom, [-3, 3, -20, 0])
    index = sstanom.mean(('longitude', 'latitude')).rolling(valid_time=3).mean('valid_time')
    return index

# import data

In [22]:
sstads = xr.open_dataset('/glade/work/acruz/Caribbean_Heat_data/ERA5/sst_anom_monthly.nc')
sstads

<xarray.Dataset> Size: 4GB
Dimensions:     (valid_time: 1032, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 8kB 1940-01-01 ... 2025-12-01
    number      (valid_time) int64 8kB ...
    expver      (valid_time) <U4 17kB ...
    month       (valid_time) int64 8kB ...
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
Data variables:
    sst         (valid_time, latitude, longitude) float32 4GB ...

# the stuff

In [23]:
oni_ds = ONI(sstads)
atl3_ds = ATL3(sstads)

In [24]:
oni_ds = oni_ds.rename({'sst': 'ONI'})

In [25]:
oni_ds = oni_ds.drop_vars(('expver', 'number', 'month'))
oni_ds

<xarray.Dataset> Size: 12kB
Dimensions:     (valid_time: 1032)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 8kB 1940-01-01 ... 2025-12-01
Data variables:
    ONI         (valid_time) float32 4kB nan nan 1.1 ... -0.3162 -0.4012 -0.4819

In [26]:
atl3_ds = atl3_ds.rename({'sst': 'ATL3'}).drop_vars(('expver', 'number', 'month'))
atl3_ds

<xarray.Dataset> Size: 12kB
Dimensions:     (valid_time: 1032)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 8kB 1940-01-01 ... 2025-12-01
Data variables:
    ATL3        (valid_time) float32 4kB nan nan 0.07213 ... 0.3336 0.3054

In [27]:
atl3_ds.attrs['long_name'] = 'ATL3 from ERA5 [°C]'
oni_ds.attrs['long_name'] = 'ONI from ERA5 [°C]'

In [28]:
oni_ds.to_netcdf('/glade/work/acruz/Caribbean_Heat_data/ERA5/ONI.nc', mode='w')
atl3_ds.to_netcdf('/glade/work/acruz/Caribbean_Heat_data/ERA5/ATL3.nc', mode='w')